<a href="https://colab.research.google.com/github/AshokHub/Gene_Plot/blob/main/genePlot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Karyotype Gene Plot**

---



This example demonstrates how to locate specific genes within a genome and visualize them in a plot. Using the [biomaRt](https://bioconductor.org/packages/biomaRt) Bioconductor package, we will automatically retrieve these gene coordinates from the Ensembl BioMart database. The resulting data frame will then be converted into a GRanges object using regioneR's `toGRanges` function. Because our target genome uses UCSC conventions, we will use `seqlevelsStyle` to adjust the chromosome naming format from Ensembl to UCSC. All examples are based on the GRCh38 assembly (_Homo sapiens_ (human) genome assembly (hg38) - Human Build 38 Organism), which aligns with the current version of BioMart.

## Installing **BiocManager**

The **BiocManager** package allows users to install and manage packages from the **BioConductor** project, including **CRAN** packages that depend or import **BioConductor** packages. Run the following commands in your R console to download the package and its dependencies:

In [ ]:
if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}

## Installing **biomaRt**

The **biomaRt** acts as an interface to **[BioMart](https://www.ensembl.org/info/data/biomart/index.html)** databases (*i.e.* **Ensembl**). Installing **biomaRt** requires R (version 4.0+) and the **BiocManager** package. Run the following commands in your R console to download the package and its dependencies:

In [ ]:
if (!require("biomaRt", quietly = TRUE)) {
    BiocManager::install("biomaRt")
}

## Installing **regioneR**

The **regioneR** offers a statistical framework based on customizable permutation tests to assess the association between genomic region sets and other genomic features. Run the following commands in your R console to download the package and its dependencies:

In [ ]:
if (!require("regioneR", quietly = TRUE)) {
    BiocManager::install("regioneR")
}

## Installing **karyoploteR**

The **karyoploteR** package is used to generate customizable, linear representations of whole genomes. It mimics standard base R plotting functions but seamlessly handles coordinate transformations, allowing users to easily overlay experimental data like BAM coverage, CNVs, and GWAS Manhattan plots along specific chromosomes. Run the following commands in your R console to download the package and its dependencies:

In [ ]:
if (!require("karyoploteR", quietly = TRUE)) {
    BiocManager::install("karyoploteR")
}

## Calling Libraries

Calling necessary libraries to convert the gene data to GRCh38 coordinates from Ensembl to UCSC and generate a chromosome-style karyotype plot.

In [ ]:
library(biomaRt)
library(regioneR)
library(GenomeInfoDb)
library(karyoploteR)

## Karyotype Plotting

In this example, the following list of genes: SPTLC3, A2MP1, MT1M, GJB1, ITIH3, GSTM1, HPD, RPL10L, GMFG, SERPINC1, CMIP, BASP1, AP1G1, PI4KB, PHF3, GOLGB1, DENND4B, ZSWIM8, MARK2, ANKRD17 are used to generate and label in the karyotype plot. Moreover, two different colors are used to differntiate up-regulated and down-regulated genes in the plot.

1. Connect to Ensembl

In [ ]:
ensembl <- useEnsembl(biomart = "ensembl", dataset = "hsapiens_gene_ensembl")

2. Define gene symbols

In [ ]:
gene.symbols <- c("SPTLC3", "A2MP1", "MT1M", "GJB1", "ITIH3", "GSTM1", "HPD",
                  "RPL10L", "GMFG", "SERPINC1", "CMIP", "BASP1", "PI4KB",
                  "PHF3", "GOLGB1", "ZSWIM8", "MARK2", "ANKRD17")

3. Fetch data frame from biomaRt

In [ ]:
bm_data <- getBM(attributes = c('chromosome_name', 'start_position',
                                'end_position', 'hgnc_symbol'),
                                filters = 'hgnc_symbol',
                                values = gene.symbols,
                                mart = ensembl)

4. Create GRanges and explicitly attach the gene symbols

In [ ]:
genes <- toGRanges(bm_data)
genes$hgnc_symbol <- bm_data$hgnc_symbol

5. Fix chromosome styles BEFORE pruning

In [ ]:
seqlevelsStyle(genes) <- "UCSC"
genes <- keepStandardChromosomes(genes, pruning.mode="coarse")

6. Plot the karyotype with auto-adjusting, non-overlapping markers

In [ ]:
group_A <- c("SPTLC3", "A2MP1", "MT1M", "GJB1", "ITIH3",
             "GSTM1", "HPD", "RPL10L", "GMFG")
genes$marker_color <- ifelse(genes$hgnc_symbol %in% group_A, "blue", "darkred")
plot_params <- getDefaultPlotParams(plot.type = 2)
plot_params$ideogramheight <- 250

7. Create the base plot

In [ ]:
kp <- plotKaryotype(genome = "hg38", plot.params = plot_params,
                    labels.plotter = NULL)
kpAddChromosomeNames(kp, col = "darkgreen", cex = 1, font = 2, family = "serif")

8. Plot markers with automatic positioning to prevent overlapping text

In [ ]:
kpPlotMarkers(kp, data = genes, labels = genes$hgnc_symbol,
              text.orientation = "horizontal", r0 = 0, r1 = 1, cex = 0.5,
              label.color = genes$marker_color,
              line.color = genes$marker_color, adjust.label.position = FALSE)